In [1]:
%reload_ext autoreload
%pylab inline
%autoreload 2
import numpy as np
from os.path import join
import sys
sys.path.insert(0, '/home/did/RTC/SMART-G/smartg/tools/')
from luts import MLUT, Idx, read_mlut
from smaccl import Smaccl, get_smac_coeffs, Ps, dPsdz
import xarray
from datetime import datetime
from netCDF4 import Dataset
from glob import glob
import h5py

Populating the interactive namespace from numpy and matplotlib
smaccl.py 1.01.00
....  testsmaccl1 class def 


In [2]:
# path to DEM
fdem = '/rfs/data/DEM/GTOPO30_DZ_MLUT.nc' 
# DEM Read (once!)
dem_lut = read_mlut(fdem) # global DEM GTOPO 30, reading once, could be long, stored into a MLUT object
# dem object is a MLUT object
#dem_lut.describe()

# SMACCL configuration</span>

In [3]:
# path to input AVHRR image (format netcdf)
#path   = '/rfs/data/AVHRR/NEW_EXAMPLE_FCDR/'
#fname_i = 'C3S-L2A-FCDR-AVHRR_NOAA-20020102205538-fv0001.nc'
#path   = '/rfs/data/C3S/VGT/EXTRACT/'
path   = '/rfs/data/VGT/VGTP_extracts_49x49_180129/'
fname_i= '1999/19990601/11_Ispra_V119990601138.h5'
fname =  path + fname_i

# GPU grid
#XBLOCK = 512
#XGRID  = 512
XBLOCK = 64
XGRID  = 64
YGRID  = 1
YBLOCK = 1

NBLOOP = 1   # eventually loops within card (for first version of smacg NLOOP=1)

aer_coef = 'CONT'  # 'CONT' or 'DES' SMAC aerosol coefficient
dir_coef_name = './' # directory containing SMAC Coefficients

### Constants
version = '1.0'

k_uh2o = 1e-1  # conversion from kg.m-2 to g.cm-2
k_uo3  = 1e-3  # (for MERRA-2, Dobson) to cm.atm
k_p0   = 1e-2  # pascal to hectopascal

# Error budget 
# (TBD, here just for tests)
Etoa   = 0     # Should come form level 1 file
ERtoa  = 0.01  # precision toa reflectance (in %)

Etaup  = 0.05 # see ATBD for post 2000 era
ERtaup = 0.15

Euo3   = 0.0
ERuo3  = 0.06 # According to Wargan et al., 2017, see ATBD

Euh2o  = 0.0
ERuh2o = 0.2  # see  Davis et al., 2017, see ATBD, no clear numbers, TBC

Epre   = 1.0  # pure meteorological uncertainy (hPa)
ERpre  = 0.0  # TBC see ATBD
  
# compile Smacg 
S=Smaccl()

....  testsmaccl1 class __init__ 
....  testsmaccl1 class __init__ exists source
... Obtain an OpenCL platform
<pyopencl.Platform 'NVIDIA CUDA' at 0x3ae4140>
... Obtain a device id
<pyopencl.Device 'GeForce GTX 660 Ti' on 'NVIDIA CUDA' at 0x3ae65b0>
... Create a context for the selected device
<pyopencl.Context at 0x3c220c0 on <pyopencl.Device 'GeForce GTX 660 Ti' on 'NVIDIA CUDA' at 0x3ae65b0>>
... Get a queue
... Build program
... Load Kernel


#  AVHRR/VGT preprocessing

In [4]:
from scipy.interpolate import RectBivariateSpline

def VGT_TIE_COMPLETE (dataset,a, size, tie):
    XSIZE, YSIZE = size
    XTIE, YTIE = tie
    ref = dataset[a] * float(dataset[a].attrs['Scale']) + float(dataset[a].attrs['Offset'])
    r1=np.arange(0,XSIZE,XTIE)
    r11,_=np.meshgrid(np.arange(XSIZE),np.arange(YSIZE))
    r2=np.arange(0,YSIZE,YTIE)
    _,r22=np.meshgrid(np.arange(XSIZE),np.arange(YSIZE))
    res = RectBivariateSpline(r1,r2,ref).ev(r11,r22)
    res = xarray.DataArray(res,dims = ('phony_dim_0', 'phony_dim_0'))
    
    dataset.drop(a)
    dataset[a] = res
    
def get_meantime(dataset):
    time = dataset.time_coverage_start
    dt1  = np.datetime64(str(time[:4]) + '-' + str(time[4:6]) + '-' + str(time[6:8]) + 'T' + \
                       str(time[9:11]) + ':' + str(time[11:13]) + ':' + str(time[13:15]))
    time = dataset.time_coverage_end
    dt2  = np.datetime64(str(time[:4]) + '-' + str(time[4:6]) + '-' + str(time[6:8]) + 'T' + \
                       str(time[9:11]) + ':' + str(time[11:13]) + ':' + str(time[13:15]))
    return ((dt2-dt1)/2. +dt1)

def to_float (dataset, elem):
    res = dataset[elem] = dataset[elem] * float(dataset[elem].attrs['Scale']) + float(dataset[elem].attrs['Offset'])
    return res

def date_to_float(d, epoch=numpy.datetime64('1980-01-01T00:00:00.000000000')):
    '''
    transform the date into a duration in minutes since epoch
    '''
    return (d - epoch).astype(float64)/1.0e9/60.

def pre(fname):
    xdataset= xarray.open_dataset(fname)
#    print xdataset
    try: 
        sensor = xdataset.sensor
    except:
        sensor = 'VGT'
        
    # sensor switch
    if sensor == 'AVHRR/3':
        conv = {'avhrr_b1':'b1','avhrr_b2':'b2','sun_zenith':'SZA', 'sun_azimuth':'SAA','sat_zenith':'VZA','sat_azimuth':'VAA'}
        xdataset.rename(conv, inplace=True)
        tab_band_internal = ['b1','b2']
        smac_coeff_name = ['NIR', 'VIS']
       
        if 'avhrr_b3a' in xdataset.data_vars.keys():
            tab_band_internal.append('b3a')
            conv = {'avhrr_b3a':'b3a'}
            xdataset.rename(conv, inplace=True)
            smac_coeff_name.append('MIR')
            
        SIZE1, SIZE2 = xdataset[tab_band_internal[0]].shape
            
#        for band in tab_band_internal:
#            if not('unc_{}'.format(band) in xdataset.data_vars.keys()):
#                void = xarray.DataArray(np.zeros((SIZE1, SIZE2), dtype='float32') + np.NaN, coords=[xdataset.Latitude,xdataset.Longitude], dims=['Latitude','Longitude'])
#                xdataset['unc_{}'.format(band)] = void

        new_attrs = {'Scale':0.01, 'Offset':0.0} #
        for band in tab_band_internal:
            xdataset[band] = xdataset[band].assign_attrs(new_attrs)
        
        platform = xdataset.platform.replace('-','')
        smac_coeff_name = ['coef_{}_{}_{}.dat'.format(platform, x, aer_coef) for x in smac_coeff_name]    
     
    elif sensor == 'VGT':
        ref = xdataset.segm_reference
        if ref[:2] == 'V1':
            sensor = 'VGT1'
        else:
            sensor = 'VGT2'
        tab_band_internal = ['B0','B2','B3','MIR']
        smac_coeff_name = ['coef_{}_{}_{}.dat'.format(sensor,str(x),aer_coef) for x in tab_band_internal]
        SIZE1, SIZE2 = xdataset[tab_band_internal[0]].shape
        XTIE, YTIE = xdataset['SZA'].shape
       
        lon0, lat0, d_lon, d_lat = [float(x) for x in xdataset.map_info.split(',')[3:7]]
        xdataset['Latitude']=lat0 - arange(SIZE1)*d_lat
        xdataset['Longitude']=lon0 + arange(SIZE2)*d_lon
        
        date_vgt = xdataset.segm_first_date
        time_vgt = xdataset.segm_first_time
        first_date = '{}T{}'.format(date_vgt, time_vgt)
        
        date_vgt = xdataset.segm_last_date
        time_vgt = xdataset.segm_last_time
        last_date = '{}T{}'.format(date_vgt, time_vgt)
        new_attrs = {'time_coverage_start':first_date,'time_coverage_end':last_date}
        xdataset = xdataset.assign_attrs(new_attrs)
        
        clm = np.ones((SIZE1, SIZE2), dtype='int8')
        sm = xdataset['SM'].data
        clear = np.where((sm&1==0) & (sm&2==0) & (sm&4==0))
        clm[clear] = 0
        xdataset['CLM'] = xarray.DataArray(clm, coords=[xdataset.Latitude,xdataset.Longitude], dims=['Latitude','Longitude'])
        
#        void = xarray.DataArray(np.zeros((SIZE1, SIZE2)) + np.NaN, coords=[xdataset.Latitude,xdataset.Longitude], dims=['Latitude','Longitude'])
#        for band in tab_band_internal:
#            xdataset['unc_{}'.format(band)] = void
            
        # preprocessing of VGT file to fill lat and lon to XSIZE and YSIZE
        # This maybe unnecessary in the future if lat and lon are given for each pixel of VGT
        if XTIE == SIZE1:
            to_float(xdataset, 'SZA')
            to_float(xdataset, 'SAA')
            to_float(xdataset, 'VZA')
            to_float(xdataset, 'VAA')
        else:
            VGT_TIE_COMPLETE(xdataset,'SZA', (SIZE1,SIZE2), (XTIE,YTIE))
            VGT_TIE_COMPLETE(xdataset,'SAA', (SIZE1,SIZE2), (XTIE,YTIE))
            VGT_TIE_COMPLETE(xdataset,'VZA', (SIZE1,SIZE2), (XTIE,YTIE))
            VGT_TIE_COMPLETE(xdataset,'VAA', (SIZE1,SIZE2), (XTIE,YTIE))

    else:
        raise("unknow sensor")
        
    lon,lat=np.meshgrid(xdataset['Longitude'],xdataset['Latitude'])
    xdataset['lat']=(('x', 'y'), lat)
    xdataset['lon']=(('x', 'y'), lon)
    # mean decimal time for the scene
    xdataset['mean-time']    = get_meantime(xdataset)
    xdataset['mean-time-dec']= date_to_float(xdataset['mean-time'].data)
    # scale ref
    for band in tab_band_internal:
        to_float(xdataset, band)
        to_float(xdataset, '{} uncertainty'.format(band))
        xdataset.rename({'{} uncertainty'.format(band):'unc_{}'.format(band)}, inplace=True)
    
    return xdataset, SIZE1, SIZE2, tab_band_internal, smac_coeff_name

data, SIZE1, SIZE2, tab_band_internal, smac_coeff_name = pre(fname)

year = str(data['mean-time'].values)[:4]
month = str(data['mean-time'].values)[5:7]
day = str(data['mean-time'].values)[8:10]

# MERRA2 preprocessing

In [5]:
# path to input ancillary MERRA 2 data according to the image date
# 1) aerosols
merra_aerosol='/rfs/data/MERRA2/aer_extinction/{0}/*MERRA2_*.tavg1_2d_aer_Nx.{0}{1}{2}*.nc4'.format(year, month, day)
print(merra_aerosol)
merra_aerosol=glob(merra_aerosol)[0]
# 2) PTWO, Pressure, Temperature, Water vapour ,Ozone
merra_ptwo='/rfs/data/MERRA2/surf_pression_water_vapor/{0}/*MERRA2_*.tavg1_2d_slv_Nx.{0}{1}{2}*.nc4'.format(year, month, day)
print(merra_ptwo)
merra_ptwo=glob(merra_ptwo)[0]

# Read MERRA2 ancillary data files and store all information into a MLUT object for further use
# (mainly for spatial and temporal interpolation)
merra = xarray.open_dataset(merra_aerosol)
merra_lut = MLUT()
# Add the good axis
merra_lut.add_axis('time', date_to_float(merra.time.data)) # float array of delta time in ns from epoch time
merra_lut.add_axis('lat',  merra.lat.data)
merra_lut.add_axis('lon',  merra.lon.data)
merra_lut.add_dataset('TOTEXTTAU', merra['TOTEXTTAU'].data, axnames=['time','lat','lon'])

merra = xarray.open_dataset(merra_ptwo)
merra_lut.add_dataset('TO3',  merra['TO3'].data,  axnames=['time','lat','lon'])
merra_lut.add_dataset('SLP',  merra['SLP'].data,  axnames=['time','lat','lon'])
merra_lut.add_dataset('T10M', merra['T10M'].data, axnames=['time','lat','lon'])
merra_lut.add_dataset('TQV',  merra['TQV'].data,  axnames=['time','lat','lon'])

# MLUT object description
merra_lut.describe()

/rfs/data/MERRA2/aer_extinction/1999/*MERRA2_*.tavg1_2d_aer_Nx.19990601*.nc4
/rfs/data/MERRA2/surf_pression_water_vapor/1999/*MERRA2_*.tavg1_2d_slv_Nx.19990601*.nc4
 Datasets:
  [0] TOTEXTTAU (float64 in [0.00475, 2.75]), axes=('time', 'lat', 'lon')
  [1] TO3 (float64 in [206, 434]), axes=('time', 'lat', 'lon')
  [2] SLP (float64 in [9.45e+04, 1.04e+05]), axes=('time', 'lat', 'lon')
  [3] T10M (float64 in [199, 322]), axes=('time', 'lat', 'lon')
  [4] TQV (float64 in [0.106, 82]), axes=('time', 'lat', 'lon')
 Axes:
  [0] time: 24 values in [10211070.0, 10212450.0]
  [1] lat: 361 values in [-90.0, 90.0]
  [2] lon: 576 values in [-180.0, 179.375]


# SMAC-G preparation of data

In [6]:
# files containg SMAC coefficients

bands_path = ["".join((dir_coef_name, 'COEFFS/'+x)) for x in smac_coeff_name]
# masking cloudy & out of orbit pixels and SZA above 90°
SM   = data['CLM'].data
SZA  = data['SZA'].data
good = np.where((SM&1 == 0) & (SM&2==0) & (SM&4==0) & (SZA < 90))
#good = np.where((SZA < 90))
GSIZE= good[0].size
NB = len(tab_band_internal)

# start with tie points
tetas       = data['SZA'].data[good].astype( float32, order='C')
tetav       = data['VZA'].data[good].astype( float32, order='C')
phis        = data['SAA'].data[good].astype(float32, order='C')
phiv        = data['VAA'].data[good].astype(float32, order='C')

lat         = data['lat'].data[good]
lon         = data['lon'].data[good]
t0          = data['mean-time-dec'].data

#interpolate merra 2 data and dem to the image location and time
taup550     = merra_lut['TOTEXTTAU']\
                [Idx(t0,  round=False, fill_value='extrema'), \
                 Idx(lat, round=False, fill_value='extrema'), \
                 Idx(lon, round=False, fill_value='extrema')] \
                .astype(float32, order='C') 
uh2o        = merra_lut['TQV']\
                [Idx(t0,  round=False, fill_value='extrema'), \
                 Idx(lat, round=False, fill_value='extrema'), \
                 Idx(lon, round=False, fill_value='extrema')] \
                .astype(float32, order='C') 
uo3         = merra_lut['TO3']\
                [Idx(t0,  round=False, fill_value='extrema'), \
                 Idx(lat, round=False, fill_value='extrema'), \
                 Idx(lon, round=False, fill_value='extrema')] \
                .astype(float32, order='C') 
p0          = merra_lut['SLP']\
                [Idx(t0,  round=False, fill_value='extrema'), \
                 Idx(lat, round=False, fill_value='extrema'), \
                 Idx(lon, round=False, fill_value='extrema')] \
                .astype(float32, order='C') 
t10m        = merra_lut['T10M']\
                [Idx(t0,  round=False, fill_value='extrema'), \
                 Idx(lat, round=False, fill_value='extrema'), \
                 Idx(lon, round=False, fill_value='extrema')] \
                .astype(float32, order='C') 
#interpolate DEM and uncertainty to the image location and time
alt         = dem_lut['elev']\
                [Idx(lat, round=False, fill_value='extrema'), \
                 Idx(lon, round=False, fill_value='extrema')] \
                .astype(float32, order='C') 
Dalt        = dem_lut['Delev']\
                [Idx(lat, round=False, fill_value='extrema'), \
                 Idx(lon, round=False, fill_value='extrema')] \
                .astype(float32, order='C')

# pressure correction for surface altitude and transformation from Pa to hPa
pressure = Ps(alt, p0*k_p0, t10m)
#ressure = Ps(np.zeros_like(t10m), p0*k_p0, t10m)
#alt     = np.zeros_like(t10m)
# quadratic mean of error due to met fields (Epre) and error due to altitude (Dalt)
pressure_err = np.sqrt((dPsdz(alt, p0*k_p0, t10m) * Dalt)**2 + Epre**2)/2.
# conversion from kg.m-2 to g.cm-2
uh2o *= k_uh2o
# conversion from Dobson to cm.atm 
uo3  *= k_uo3

# prepare radiometry array
rtoa        = np.zeros((NB, GSIZE), dtype='float32', order='C')
rtoa_err    = np.zeros((NB, GSIZE), dtype='float32', order='C')
for iband, band in enumerate(tab_band_internal):
    rtoa[iband,:]     = data[band].data[good]
    rtoa_err[iband,:] = data['unc_' +   band].data[good]

## Input arrays resizing to adapt to GPU grid size
#
# number of planes in the 3rd dimension to be created
Z = int(math.ceil(float(GSIZE)/float(XBLOCK*XGRID)))
GSIZEXT = Z * XBLOCK * XGRID

# the "ext" suffix is for extended arrays, larger than the good pixels size, it is completed by NaN's
rtoa_ext     = np.zeros((NB, GSIZEXT), dtype='float32') + np.NaN
taup550_ext  = np.zeros((GSIZEXT), dtype='float32') + np.NaN
uo3_ext      = np.zeros((GSIZEXT), dtype='float32') + np.NaN
pressure_ext = np.zeros((GSIZEXT), dtype='float32') + np.NaN
uh2o_ext     = np.zeros((GSIZEXT), dtype='float32') + np.NaN
tetas_ext    = np.zeros((GSIZEXT), dtype='float32') + np.NaN
tetav_ext    = np.zeros((GSIZEXT), dtype='float32') + np.NaN
phis_ext     = np.zeros((GSIZEXT), dtype='float32') + np.NaN
phiv_ext     = np.zeros((GSIZEXT), dtype='float32') + np.NaN
for i in range(NB):
    rtoa_ext[i,:GSIZE] = rtoa[i,:]
taup550_ext[:GSIZE]  = taup550
uo3_ext[:GSIZE]      = uo3
pressure_ext[:GSIZE] = pressure
uh2o_ext[:GSIZE]     = uh2o
tetas_ext[:GSIZE]    = tetas
tetav_ext[:GSIZE]    = tetav
phis_ext[:GSIZE]     = phis
phiv_ext[:GSIZE]     = phiv

# Getting SMAC coefficients
coeffs     = get_smac_coeffs(bands_path)

# Processing by the GPU

In [7]:
%%time
# Input arrays reshaping
rtoa_ext     = np.reshape(rtoa_ext,    (NB,Z,XBLOCK,XGRID), order='F')
tetas_ext    = np.reshape(tetas_ext,   (Z,XBLOCK,XGRID),    order='F')
tetav_ext    = np.reshape(tetav_ext,   (Z,XBLOCK,XGRID),    order='F')
phis_ext     = np.reshape(phis_ext,    (Z,XBLOCK,XGRID),    order='F')
phiv_ext     = np.reshape(phiv_ext,    (Z,XBLOCK,XGRID),    order='F')
uh2o_ext     = np.reshape(uh2o_ext,    (Z,XBLOCK,XGRID),    order='F')
uo3_ext      = np.reshape(uo3_ext,     (Z,XBLOCK,XGRID),    order='F')
taup550_ext  = np.reshape(taup550_ext, (Z,XBLOCK,XGRID),    order='F')
pressure_ext = np.reshape(pressure_ext,(Z,XBLOCK,XGRID),    order='F')

# Run
(rsurf_ext,Jrtoa_ext,Juo3_ext,Juh2o_ext,Jpre_ext,Jtaup_ext) = S.run(coeffs, tetas_ext, tetav_ext, 
        phis_ext, phiv_ext, uh2o_ext, uo3_ext, taup550_ext, pressure_ext, rtoa_ext,
        XBLOCK=XBLOCK, XGRID=XGRID, NBLOOP=NBLOOP)

# Output arrays reshaping 
rsurf_ext = np.reshape(rsurf_ext,(NB,GSIZEXT), order='F')
Jrtoa_ext = np.reshape(Jrtoa_ext,(NB,GSIZEXT), order='F')
Juo3_ext  = np.reshape(Juo3_ext, (NB,GSIZEXT), order='F')
Juh2o_ext = np.reshape(Juh2o_ext,(NB,GSIZEXT), order='F')
Jpre_ext  = np.reshape(Jpre_ext, (NB,GSIZEXT), order='F')
Jtaup_ext = np.reshape(Jtaup_ext,(NB,GSIZEXT), order='F')

....  testsmaccl1 class run 
Data points: 16384
Workers: 32
.... Smaccl: Smaccl  kernel (run) begin  
Global size: (16384,)
Local  size: (64,)
64 64
Execution time of test: 0.0132588 s
.... Smaccl: Smaccl  kernel (run) end  
CPU times: user 28.1 ms, sys: 17.6 ms, total: 45.8 ms
Wall time: 67.8 ms


# Post Processing for uncertainties and output writing

In [15]:
rsurf  = np.zeros((NB,SIZE1,SIZE2))
Drsurf = np.zeros((NB,SIZE1,SIZE2))
inter  = np.zeros((SIZE1,SIZE2)) + np.nan
stock  = np.zeros((GSIZE))
BREAKPOINT = False
INPUT      = False

if BREAKPOINT :
    Jrtoa  = np.zeros((NB,SIZE1,SIZE2))
    Juo3   = np.zeros((NB,SIZE1,SIZE2))
    Juh2o  = np.zeros((NB,SIZE1,SIZE2))
    Jpre   = np.zeros((NB,SIZE1,SIZE2))
    Jtaup  = np.zeros((NB,SIZE1,SIZE2))
    Drtoa  = np.zeros((NB,SIZE1,SIZE2))
    Duo3   = np.zeros((NB,SIZE1,SIZE2))
    Duh2o  = np.zeros((NB,SIZE1,SIZE2))
    Dpre   = np.zeros((NB,SIZE1,SIZE2))
    Dtaup  = np.zeros((NB,SIZE1,SIZE2))
    
if INPUT :
    Irtoa  = np.zeros((NB,SIZE1,SIZE2))
    Iuo3   = np.zeros((SIZE1,SIZE2)) + np.nan
    Iuh2o  = np.zeros((SIZE1,SIZE2)) + np.nan
    Ipre   = np.zeros((SIZE1,SIZE2)) + np.nan
    Itaup  = np.zeros((SIZE1,SIZE2)) + np.nan
    Ialt   = np.zeros((SIZE1,SIZE2)) + np.nan
    Iuo3[good]   = uo3
    Iuh2o[good]  = uh2o
    Ipre[good]   = pressure
    Itaup[good]  = taup550
    #Ialt[good]   = alt

for i in range(NB): 

    inter[good]  = rsurf_ext[i,:GSIZE]
    rsurf[i,:,:] = inter
    
    if INPUT:
        inter[good] = rtoa[i,:]
        Irtoa[i,:,:]= inter    
    
    inter[good]  = abs(Jrtoa_ext[i,:GSIZE] * rtoa_err[i,:]               )
    if BREAKPOINT: Drtoa[i,:,:] = inter
    stock        = inter[good]**2
    inter[good]  = abs(Jtaup_ext[i,:GSIZE] * (Etaup + ERtaup * taup550  ))
    if BREAKPOINT: Dtaup[i,:,:] = inter
    stock       += inter[good]**2 
    inter[good]  = abs(Juo3_ext[i,:GSIZE]  * (Euo3  + ERuo3  * uo3      ))
    if BREAKPOINT: Duo3[i,:,:]  = inter
    stock       += inter[good]**2 
    inter[good]  = abs(Juh2o_ext[i,:GSIZE] * (Euh2o + ERuh2o * uh2o     ))
    if BREAKPOINT: Duh2o[i,:,:] = inter
    stock       += inter[good]**2 
    inter[good]  = abs(Jpre_ext[i,:GSIZE] *  pressure_err)
    if BREAKPOINT: Dpre[i,:,:]  = inter
    stock       += inter[good]**2 
    
    inter[good]  = np.sqrt(stock/5.)
    Drsurf[i,:,:]= inter
    
    if BREAKPOINT:
        inter[good]  = Jrtoa_ext[i,:GSIZE]
        Jrtoa[i,:,:] = inter
        inter[good]  = Juo3_ext[i,:GSIZE]
        Juo3[i,:,:]  = inter
        inter[good]  = Juh2o_ext[i,:GSIZE]
        Juh2o[i,:,:] = inter
        inter[good]  = Jpre_ext[i,:GSIZE]
        Jpre[i,:,:]  = inter
        inter[good]  = Jtaup_ext[i,:GSIZE]
        Jtaup[i,:,:] = inter
        
del inter
del stock
# output file creation        
#fname = path + fname_o
#fname = '/rfs/user/bruno/C3S/test_cl.h5'
fname = '/rfs/user/bruno/smac/output/11_Ispra_V119990601138_cl.h5'
out = h5py.File(fname, 'w')

for att, value in data.attrs.items():
    out.attrs[att] = value
    
out.attrs['date_created'] = str(datetime.now())
out.attrs['production_center'] = 'hygeos'
out.attrs['version'] = version

out.create_dataset('Latitude', data['SZA'].shape, dtype='float32', compression='gzip', compression_opts=9)
out['Latitude'].attrs['Unit'] = 'degree'
out.create_dataset('Longitude', data['SZA'].shape, dtype='float32', compression='gzip', compression_opts=9)
out['Longitude'].attrs['Unit'] = 'degree'

out.create_dataset('SZA', data['SZA'].shape, dtype='float32', compression='gzip', compression_opts=9)
out['SZA'].attrs['Unit'] = 'degree'
out.create_dataset('SAA', data['SAA'].shape, dtype='float32', compression='gzip', compression_opts=9)
out['SAA'].attrs['Unit'] = 'degree'
out.create_dataset('VZA', data['VZA'].shape, dtype='float32', compression='gzip', compression_opts=9)
out['VZA'].attrs['Unit'] = 'degree'
out.create_dataset('VAA', data['VAA'].shape, dtype='float32', compression='gzip', compression_opts=9)
out['VAA'].attrs['Unit'] = 'degree'
out.create_dataset('SM', data['SM'].shape, dtype='float32', compression='gzip', compression_opts=9)
out['SM'].attrs['Unit'] = 'degree'

dataset_names = ['TOC Blue','TOC Red', 'TOC NIR', 'TOC SWIR']
for idx in range(4):
    band = dataset_names[idx]
    out.create_dataset(band, rsurf[idx].shape, dtype='float32', compression='gzip', compression_opts=9)
    out[band].attrs['long_name'] = 'Top of Canopy Reflectance'
    out[band].attrs['unit'] = 'None'
    band = '{} error'.format(band)
    out.create_dataset(band, rsurf[idx].shape, dtype='float32', compression='gzip', compression_opts=9)
    out[band].attrs['long_name'] = 'Uncertainty Top of Canopy Reflectance'
    out[band].attrs['unit'] = 'None'


if BREAKPOINT:
    out.create_dataset('Drtoa', Drtoa.shape, dtype='float32', compression='gzip', compression_opts=9)
    out.create_dataset('Duo3', Duo3.shape, dtype='float32', compression='gzip', compression_opts=9)
    out.create_dataset('Duh2o', Duh2o.shape, dtype='float32', compression='gzip', compression_opts=9)
    out.create_dataset('Dpre', Dpre.shape, dtype='float32', compression='gzip', compression_opts=9)
    out.create_dataset('Dtaup', Dtaup.shape, dtype='float32', compression='gzip', compression_opts=9)
    
if INPUT:
    out.create_dataset('rtoa', Irtoa.shape, dtype='float32', compression='gzip', compression_opts=9)
    out.create_dataset('uo3',  Iuo3.shape, dtype='float32', compression='gzip', compression_opts=9)
    out.create_dataset('uh2o', Iuh2o.shape, dtype='float32', compression='gzip', compression_opts=9)
    out.create_dataset('pre', Ipre.shape, dtype='float32', compression='gzip', compression_opts=9)
    out.create_dataset('taup', Itaup.shape, dtype='float32', compression='gzip', compression_opts=9)
    #out.create_dataset('alt', Ialt.shape, dtype='float32', compression='gzip', compression_opts=9)

In [16]:
for idx in range(4):
    band = dataset_names[idx]
    out[band][:]  = rsurf[idx]
    band = '{} error'.format(band)
    out[band][:] = Drsurf[idx]
    
lon, lat = np.meshgrid(data['Longitude'].data, data['Latitude'].data)
out['Latitude'][:]  = lat
out['Longitude'][:] = lon

out['SZA'][:] = data['SZA'].data
out['SAA'][:] = data['SAA'].data
out['VZA'][:] = data['VZA'].data
out['VAA'][:] = data['VAA'].data
out['SM'][:] = data['SM'].data

if BREAKPOINT:
    out['Drtoa'][:] = Drtoa
    out['Duo3'][:]  = Duo3
    out['Duh2o'][:] = Duh2o
    out['Dpre'][:]  = Dpre
    out['Dtaup'][:] = Dtaup
    
if INPUT:
    out['rtoa'][:]  = Irtoa
    out['uo3'][:]   = Iuo3
    out['uh2o'][:]  = Iuh2o
    out['pre'][:]   = Ipre
    out['taup'][:]  = Itaup
    #out['alt'][:]   = Ialt

In [17]:
out.close()

# netcdf output writing

In [ ]:
rsurf  = np.zeros((NB,SIZE1,SIZE2))
Drsurf = np.zeros((NB,SIZE1,SIZE2))
inter  = np.zeros((SIZE1,SIZE2)) + np.nan
stock  = np.zeros((GSIZE))

for i in range(NB): 

    inter[good]  = rsurf_ext[i,:GSIZE]
    rsurf[i,:,:] = inter
    
    inter[good]  = abs(Jrtoa_ext[i,:GSIZE] * rtoa_err[i,:]               )
    stock        = inter[good]**2
    inter[good]  = abs(Jtaup_ext[i,:GSIZE] * (Etaup + ERtaup * taup550  ))
    stock       += inter[good]**2 
    inter[good]  = abs(Juo3_ext[i,:GSIZE]  * (Euo3  + ERuo3  * uo3      ))
    stock       += inter[good]**2 
    inter[good]  = abs(Juh2o_ext[i,:GSIZE] * (Euh2o + ERuh2o * uh2o     ))
    stock       += inter[good]**2 
    inter[good]  = abs(Jpre_ext[i,:GSIZE] *  pressure_err)
    stock       += inter[good]**2 
    
    inter[good]  = np.sqrt(stock/5.)
    Drsurf[i,:,:]= inter


fname = '/rfs/user/bruno/smac/output/11_Ispra_V119990601138_cl.nc'

out = Dataset(fname, 'w', format='NETCDF4')

for att, value in data.attrs.items():
    out.setncattr(att,value)

out.date_created = str(datetime.now())
out.production_center = 'hygeos'
out.version = version

width = rsurf.shape[1]
height = rsurf.shape[2]
h = out.createDimension('height', height)
w = out.createDimension('width', width)

dataset_names = ['TOC Blue','TOC Red', 'TOC NIR', 'TOC SWIR']
for idx in range(4):
    sds = out.createVariable(dataset_names[idx], 'f', ('height','width'), complevel=9)
    sds[:] = rsurf[idx]
    sds.long_name = 'Top of Canopy Reflectance'
    sds.unit = 'None'
    sds = out.createVariable('{} error'.format(dataset_names[idx]), 'f', ('height','width'), complevel=9)
    sds[:] = Drsurf[idx]
    sds.long_name = 'Uncertainty Top of Canopy Reflectance'
    sds.unit = 'None'

    
lon, lat = np.meshgrid(data['Longitude'].data, data['Latitude'].data)

sds = out.createVariable('Lat', 'f', ('height','width'), complevel=9)
sds[:] = lat
sds.unit = 'Degree'

sds = out.createVariable('Lon', 'f', ('height','width'), complevel=9)
sds[:] = lon
sds.unit = 'Degree'

sds = out.createVariable('SZA', 'f', ('height','width'), complevel=9)
sds.unit = 'Degree'
sds[:] = data['SZA'].data

sds = out.createVariable('SAA', 'f', ('height','width'), complevel=9)
sds[:] = data['SAA'].data
sds.unit = 'Degree'

sds = out.createVariable('VZA', 'f', ('height', 'width'), complevel=9)
sds[:] = data['VZA'].data
sds.unit = 'Degree'

sds = out.createVariable('VAA', 'f', ('height', 'width'), complevel=9)
sds[:] = data['VAA'].data
sds.unit = 'Degree'

sds = out.createVariable('SM', 'i', ('height', 'width'), complevel=9)
sds[:] = data['SM'].data

out.close()

# Some Plots

In [ ]:
sys.path.append('/home/did/MERIS/BRDF/Fourth_Reprocessing/')
from geoutils.figures import Figures
fig1 = Figures(cols=4, rows=5, size=(4,4), fontsize=18)

for k in range(4): 
    fig1.imshow(Jrtoa[k,:,:],vmin= 0, vmax=2, title=r'$\mid J^{\rho_{toa}}_{\rho_{toc}}\mid ' + tab_band_internal[k] + '$' ,  shrink=0.6, colorbar=True)
for k in range(4): 
    fig1.imshow(Juo3[k,:,:], vmin= 0, vmax=.05, title=r'$\mid J^{U_{O_3}}_{\rho_{toc}}\mid$' ,  shrink=0.6, colorbar=True)
for k in range(4): 
    fig1.imshow(Juh2o[k,:,:], vmin= 0, vmax=.01, title=r'$\mid J^{U_{H_2O}}_{\rho_{toc}}\mid$' ,  shrink=0.6, colorbar=True)
for k in range(4): 
    fig1.imshow(abs(Jpre[k,:,:]*1000), vmin= 0, vmax=0.2, title=r'$\mid J^{P_S}_{\rho_{toc}}\mid (\times 10^3)$' ,  shrink=0.6, colorbar=True)
for k in range(4): 
    fig1.imshow(abs(Jtaup[k,:,:]), vmin= 0, vmax=0.2, title=r'$\mid J ^{\tau_a^{550}}_{\rho_{toc}}\mid$',  shrink=0.6, colorbar=True)

In [ ]:
fig2 = Figures(cols=4, rows=7, size=(4,4), fontsize=18)
vmax = 0.01

for k in range(4):    
    fig2.imshow(rsurf[k,:,:], vmin= 0, vmax=0.3,  title=r'$\rho_{toc} ' + tab_band_internal[k] + '$',  shrink=0.6, colorbar=True)
for k in range(4):    
    fig2.imshow(Drsurf[k,:,:],vmin= 0, vmax=vmax, title=r'$\Delta\rho_{toc}$ ' ,  shrink=0.6, colorbar=True)
for k in range(4): 
    fig2.imshow(Drtoa[k,:,:], vmin= 0, vmax=vmax, title=r'$\Delta^{\rho_{toa}}_{\rho_{toc}}$' ,  shrink=0.6, colorbar=True)
for k in range(4): 
    fig2.imshow(Duo3[k,:,:],  vmin= 0, vmax=vmax, title=r'$\Delta^{U_{O_3}}_{\rho_{toc}}$' ,  shrink=0.6, colorbar=True)
for k in range(4): 
    fig2.imshow(Duh2o[k,:,:], vmin= 0, vmax=vmax, title=r'$\Delta^{U_{H_2O}}_{\rho_{toc}}$' ,  shrink=0.6, colorbar=True)
for k in range(4): 
    fig2.imshow(Dpre[k,:,:],  vmin= 0, vmax=vmax, title=r'$\Delta^{P_S}_{\rho_{toc}} $' ,  shrink=0.6, colorbar=True)
for k in range(4): 
    fig2.imshow(Dtaup[k,:,:], vmin= 0, vmax=vmax, title=r'$\Delta^{\tau_a^{550}}_{\rho_{toc}}$',  shrink=0.6, colorbar=True)

In [ ]:
fig3 = Figures(cols=4, rows=2, size=(4,4), fontsize=18)
vmax = 0.01
for k in range(4):    
    fig3.imshow(Irtoa[k,:,:], vmin= 0, vmax=0.3, title=r'$\rho_{toa} ' + tab_band_internal[k] + '$',  shrink=0.6, colorbar=True)
fig3.imshow(Iuo3[:,:],   title=r'$U_{O_3}$ ',   shrink=0.6, colorbar=True)
fig3.imshow(Iuh2o[:,:],  title=r'$U_{H_2O}$ ',  shrink=0.6, colorbar=True)
fig3.imshow(Ipre[:,:],  title=r'$P_{S}$ ',  shrink=0.6, colorbar=True)
fig3.imshow(Itaup[:,:],  title=r'$\tau_a^{550}$ ',  shrink=0.6, colorbar=True)